# DumpWatch 2.0 Task B — Retrospective comparison

This notebook compares a reference image-difference rule, the frozen Real Only SVM and a compact MLP. The real holdout was previously evaluated. Results cannot establish prospective performance. Task A remains unchanged.

Set the evidence folder and authoritative ZIP below. The retrospective stage defaults to disabled. For the recorded run the user authorized retrospective scoring. All output goes to a new directory.

In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get("DUMPWATCH_PACKAGE", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "project_setup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within the unpacked code folder or set DUMPWATCH_PACKAGE to it. See SETUP.md.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from project_setup import project_paths, new_output, enable_demo_vendor
P = project_paths()
ARCHIVE_HASH = "1e97cfe51b4ef7fc9e61aaa7cbd2cbce4ac5a24fd274f07503d7deb5138f0412"
PINNED = {'real_location_split.csv': '0ae3ff815747ba5c8c677fb3670c373aecbe9ecf2b4e3bd6b2410450748a3e60', 'frozen_train_development_pair_inventory.csv': 'bdbb8bca0ce9d8daf398f2ac613090ac2a39be570d05f620733e5ff40c283469', 'frozen_test_pair_inventory.csv': 'c44ce60a76f51bbcb19873499b27ee3cb4a3adc96326b2455dc87d77edbae004', 'frozen_test_inventory_lock.json': '62b32e7b45cc49cb2f6eefbb0aa1ce1c24dec9e0ead6bb002340a7c17a8c950f', 'frozen_real_only_svm.json': 'b6bc6391b3af0e7ea2999e0702876c1afa980f51f83470fcce4238b430f0b6b5', 'original_development_replay.csv': '32854527bb71747534f99ae8d2f138b2ab12608a81794bf5fce5e756572fad54', 'results/final_test_predictions.csv': '1118bba88e9ea85e0f55b8254fca17d8d0d738473c0c360d97c53de4cd75fef8'}
from pathlib import Path
import os
import csv
import json
import hashlib
import io
import zipfile
from collections import Counter
from importlib.metadata import version
from inspect import signature
from datetime import datetime, timezone

import numpy as np
from PIL import Image, ImageDraw, ImageOps
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from threadpoolctl import threadpool_limits

# Set these environment variables or edit the defaults for your local/Colab copy.
EVIDENCE = Path(os.environ.get("DUMPWATCH_EVIDENCE", ROOT / "Additional_Evidence/Task_B_Model_Comparison/evidence_inputs"))
ARCHIVE = P["real_zip"]
RUN = Path(os.environ["DUMPWATCH_COMPARISON_RUN"]) if os.environ.get("DUMPWATCH_COMPARISON_RUN") else new_output("mlp_comparison") / "run"
RUN_HOLDOUT = os.environ.get("DUMPWATCH_RETROSPECTIVE", "0") == "1"
LABELS = ["waste_still_present", "fully_cleared"]
SETTINGS = dict(hidden_layer_sizes=(2,), activation="tanh", solver="lbfgs", alpha=1.0,
                random_state=37, tol=1e-4, max_iter=2000, max_fun=15000)
EXPECTED = {
    "training": ({"01","03","05","06","07","09"}, (14,6)),
    "development": ({"02","08"}, (6,2)),
    "retrospective": ({"04","10"}, (6,3)),
}

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha(raw):
    return hashlib.sha256(raw).hexdigest()

def rows(relative):
    with (EVIDENCE / relative).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

def save_json(name, obj):
    with (RUN / name).open("x", encoding="utf8") as stream:
        json.dump(obj, stream, indent=2, allow_nan=False)
        stream.write("\n")

def save_csv(name, records):
    with (RUN / name).open("x", encoding="utf8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)


Validate pinned inputs, original split, approved pair IDs and labels, archive metadata, and frozen SVM records. No held-out image pixels are loaded here.

In [ ]:
require("sample_weight" in signature(MLPClassifier.fit).parameters, "MLPClassifier with weighted fitting is required")
require(sha(ARCHIVE.read_bytes()) == ARCHIVE_HASH, "Authoritative ZIP differs")
for file, digest in PINNED.items():
    require(sha((EVIDENCE / file).read_bytes()) == digest, f"Evidence changed: {file}")

splits = {r["location_id"]: r["split"] for r in rows("real_location_split.csv")}
groups = {}
for group, (sites, counts) in EXPECTED.items():
    split = dict(training="train", development="validation", retrospective="test")[group]
    require({loc[-2:] for loc, name in splits.items() if name == split} == sites, f"Split changed: {group}")

for row in rows("frozen_train_development_pair_inventory.csv"):
    if row["status"] == "usable":
        groups.setdefault(row["split"], []).append(dict(row, ground_truth=row["target"]))
groups["retrospective"] = rows("frozen_test_pair_inventory.csv")

with zipfile.ZipFile(ARCHIVE) as source:
    members = set(source.namelist())
    manifest = {r["relative_path"]: r for r in csv.DictReader(
        io.StringIO(source.read("dataset_manifest.csv").decode("utf-8-sig")))}
all_ids = []
partition_hashes = {}
for group, (sites, counts) in EXPECTED.items():
    subset = groups[group]
    require(Counter(r["ground_truth"] for r in subset) == dict(zip(LABELS, counts)), f"Bad labels/counts: {group}")
    require({r["location_id"][-2:] for r in subset} == sites, f"Bad site membership: {group}")
    require(len({r["inspection_sha256"] for r in subset}) == len(subset), f"Duplicated inspection: {group}")
    partition_hashes[group] = set()
    for row in subset:
        all_ids.append(row["pair_id"])
        stage = row["inspection_stage"]
        require((stage,row["ground_truth"]) in {("A_partial",LABELS[0]),("A_removed",LABELS[1])}, "Invalid outcome")
        require(row["W_sha256"] != row["inspection_sha256"], "Copied inspection")
        require(row["pair_id"] == f'{row["location_id"][-2:]}_{row["case_id"][5:7]}_{stage}', "Incorrect pair ID")
        for key, semantic in (("W","W"),("inspection",stage)):
            path = row[key+"_path"]
            require(path in members and path in manifest, f"Missing verified member: {path}")
            meta = manifest[path]
            digest = row[key+"_sha256"]
            require((meta["location_id"],meta["case_id"],meta["semantic_stage"],meta["file_hash"]) ==
                    (row["location_id"],row["case_id"],semantic,digest), f"Role or provenance mismatch: {path}")
            partition_hashes[group].add(digest)
require(len(all_ids) == len(set(all_ids)) == 37, "Pair IDs must be unique")
for a, hashes in partition_hashes.items():
    for b in partition_hashes:
        require(a == b or not hashes.intersection(partition_hashes[b]), "Cross-partition duplicate")

dev_svm = {r["pair_id"]:r for r in rows("original_development_replay.csv")}
test_svm = {r["pair_id"]:r for r in rows("results/final_test_predictions.csv")}
for group, saved, truth, loc in (("development",dev_svm,"truth","location"),
                                 ("retrospective",test_svm,"ground_truth","location_id")):
    require(set(saved) == {r["pair_id"] for r in groups[group]}, "SVM IDs differ from locked IDs")
    for row in groups[group]:
        recorded = saved[row["pair_id"]]
        require((recorded[truth], recorded[loc]) == (row["ground_truth"],row["location_id"]), "SVM row mismatch")
        if group == "retrospective":
            require(all(recorded[k] == v for k,v in row.items()), "Frozen holdout row changed")
print("Verified pair membership, labels and ZIP member availability:", {k:len(v) for k,v in groups.items()})


Fit and serialize the training-only reference cutoff, majority class, and four-input/two-hidden-unit L2 MLP. The saved lock precedes any held-out image decoding. Tenable comparisons preserve case weighting and frozen crop coordinates.

In [ ]:
def crop(source, row, key):
    raw = source.read(row[key+"_path"])
    require(sha(raw) == row[key+"_sha256"], f"Image changed: {row[key+'_path']}")
    l,t,r,b = map(float, row[key+"_roi_xyxy"].split())
    require(0 <= l < r <= 1 and 0 <= t < b <= 1, "Invalid ROI")
    with Image.open(io.BytesIO(raw)) as im:
        rgb = im.convert("RGB")
    if key+"_decoded_rgb_sha256" in row:
        digest = sha(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())
        require(digest == row[key+"_decoded_rgb_sha256"], "Decoded RGB differs")
    w,h = rgb.size
    patch = rgb.crop((round(l*w),round(t*h),round(r*w),round(b*h)))
    require(min(patch.size) >= 64, "Crop is too small")
    return np.asarray(patch.resize((128,128),Image.Resampling.BILINEAR),dtype=np.float32)/255

def features(w, a):
    diff = np.abs(w-a)
    blocks = diff.reshape(8,16,8,16,3).mean(axis=(1,3,4)).ravel()
    def edge(image):
        gray = image @ np.array([.299,.587,.114],dtype=np.float32)
        dy,dx = np.gradient(gray)
        return float(np.hypot(dx,dy).mean())
    now = edge(a)
    return [float(diff.mean()),float(np.sort(blocks)[-4:].mean()),now,now-edge(w)]

def extract(group):
    with zipfile.ZipFile(ARCHIVE) as source:
        return np.asarray([features(crop(source,row,"W"),crop(source,row,"inspection"))
                           for row in groups[group]],dtype=np.float64)

def weighted_median(values, weights):
    order = np.argsort(values, kind="stable")
    k = np.searchsorted(np.cumsum(weights[order]), weights.sum()/2)
    return float(values[order[k]])

x = extract("training")
training = groups["training"]
y = np.array([LABELS.index(r["ground_truth"]) for r in training])
case_counts = Counter((r["location_id"],r["case_id"]) for r in training)
weights = np.array([1/case_counts[r["location_id"],r["case_id"]] for r in training])
weights /= weights.mean()
medians = [weighted_median(x[y==k,0],weights[y==k]) for k in range(2)]
majority = int(np.bincount(y).argmax())
mean = np.average(x,axis=0,weights=weights)
variance = np.average((x-mean)**2,axis=0,weights=weights)
scale = np.where(variance==0,1,np.sqrt(variance))
model = MLPClassifier(**SETTINGS)
with threadpool_limits(limits=1):
    model.fit((x-mean)/scale,y,sample_weight=weights)
params = [*model.coefs_,*model.intercepts_]
require(model.classes_.tolist()==[0,1] and sum(a.size for a in params)==13
        and all(np.isfinite(a).all() for a in params) and np.isfinite(model.loss_), "Invalid fitted model")
require(model.n_iter_ < SETTINGS["max_iter"], "MLP hit iteration limit")
frozen = dict(settings=SETTINGS, feature_names=["RGB_MAE","top4_block_RGB_MAE",
              "inspection_gradient_mean","inspection_minus_deposit_gradient_mean"],
              class_order=LABELS, class_training_mae_medians=medians, difference_threshold=sum(medians)/2,
              majority=LABELS[majority], mean=mean.tolist(), scale=scale.tolist(),
              coefficients=[a.tolist() for a in model.coefs_],
              intercepts=[a.tolist() for a in model.intercepts_],
              iterations=int(model.n_iter_), loss=float(model.loss_),
              training_ids=[r["pair_id"] for r in training], dependencies={
              k:version(k) for k in ("numpy","Pillow","scikit-learn","scipy","threadpoolctl")})
RUN.mkdir(parents=True,exist_ok=False)
save_json("frozen_new_methods.json",frozen)
model_hash = sha((RUN/"frozen_new_methods.json").read_bytes())
save_json("pre_holdout_lock.json", dict(model_sha256=model_hash, archive_sha256=ARCHIVE_HASH,
          evidence_sha256=PINNED, utc=datetime.now(timezone.utc).isoformat(),
          holdout_enabled=RUN_HOLDOUT, scope="Previously evaluated holdout. No new model selection."))
print("Frozen new methods:",model_hash)
print("Training class MAE medians:",medians,"; MLP iterations:",model.n_iter_)


Run the fixed methods on development, and on the previously evaluated holdout if enabled. Reuse saved SVM predictions. Undefined class precision is saved as null; zero F1 is used when no positive decisions are made for a supported class.

In [ ]:
METHODS = ["Image Difference","Frozen Real Only SVM","Compact MLP","Training Majority"]

def compare(group, saved):
    locked_path = RUN/"frozen_new_methods.json"
    raw = locked_path.read_bytes()
    require(sha(raw) == model_hash, "New model changed after lock")
    spec = json.loads(raw)
    values = extract(group)
    standardized = (values-spec["mean"])/spec["scale"]
    w1,w2 = map(np.asarray,spec["coefficients"])
    b1,b2 = map(np.asarray,spec["intercepts"])
    logits = (np.tanh(standardized @ w1 + b1) @ w2 + b2).ravel()
    require(np.isfinite(logits).all(), "Invalid MLP logits")
    results = []
    for i,row in enumerate(groups[group]):
        previous = saved[row["pair_id"]]
        historical_forced = previous.get("forced_prediction", previous.get("forced_diagnostic_prediction"))
        historical_margin = previous.get("decision_score",previous.get("score"))
        predictions = [
            (values[i,0], LABELS[int(values[i,0]>=spec["difference_threshold"])], ""),
            (float(historical_margin),historical_forced,previous["fixed_prediction"]),
            (logits[i],LABELS[int(logits[i]>=0)],""),
            (None,spec["majority"],""),
        ]
        for method,(score,pred,fixed) in zip(METHODS,predictions):
            results.append(dict(partition=group,pair_id=row["pair_id"],location=row["location_id"],
                 case=row["case_id"],inspection_stage=row["inspection_stage"],truth=row["ground_truth"],
                 method=method,score="" if score is None else float(score),prediction=pred,
                 fixed_svm_prediction=fixed))
    return results

def metrics(records, key="prediction", selective=False):
    actual = [r["truth"] for r in records]
    predicted = [r[key] for r in records]
    labels = LABELS + (["abstain"] if selective else [])
    cm = confusion_matrix(actual,predicted,labels=labels)[:2].tolist()
    n = len(actual)
    accepted = sum(sum(r[:2]) for r in cm)
    correct = cm[0][0]+cm[1][1]
    result = dict(n=n, row_labels=LABELS, column_labels=labels, matrix=cm,
                  correct=correct, accepted=accepted, abstentions=n-accepted,
                  coverage=accepted/n, accuracy=correct/accepted if accepted else None,
                  false_clearances=cm[0][1], actual_present=sum(cm[0]),
                  false_clearance_rate=cm[0][1]/sum(cm[0]) if sum(cm[0]) else None,
                  false_alarms=cm[1][0],actual_cleared=sum(cm[1]),
                  false_alarm_rate=cm[1][0]/sum(cm[1]) if sum(cm[1]) else None)
    if not selective:
        p,r,f,s = precision_recall_fscore_support(actual,predicted,labels=LABELS,zero_division=0)
        result["macro_f1"]=float(f.mean())
        result["classes"]={label:dict(precision=float(p[i]) if any(p==label for p in predicted) else None,
                                      recall=float(r[i]) if s[i] else None,f1=float(f[i]),support=int(s[i]))
                           for i,label in enumerate(LABELS)}
    return result

scored = {"development":compare("development",dev_svm)}
if RUN_HOLDOUT:
    scored["retrospective"] = compare("retrospective",test_svm)
else:
    print("Retrospective scoring disabled. Set DUMPWATCH_RETROSPECTIVE=1 only for authorized reproduction.")
report = {}
flat = []
for group,records in scored.items():
    save_csv(group+"_predictions.csv", records)
    report[group] = {}
    for method in METHODS:
        subset = [r for r in records if r["method"] == method]
        stat = metrics(subset)
        report[group][method] = stat
        flat.append(dict(partition=group,method=method,correct=stat["correct"],n=stat["n"],
                    accuracy=stat["accuracy"],macro_f1=stat["macro_f1"],
                    present_precision=stat["classes"][LABELS[0]]["precision"],
                    present_recall=stat["classes"][LABELS[0]]["recall"],
                    cleared_precision=stat["classes"][LABELS[1]]["precision"],
                    cleared_recall=stat["classes"][LABELS[1]]["recall"],
                    false_clearances=stat["false_clearances"],actual_present=stat["actual_present"],
                    false_clearance_rate=stat["false_clearance_rate"],
                    false_alarms=stat["false_alarms"],actual_cleared=stat["actual_cleared"]))
    report[group]["Frozen SVM fixed policy"] = metrics(
        [r for r in records if r["method"]==METHODS[1]],"fixed_svm_prediction",True)

save_json("metrics.json",report)
save_csv("comparison_metrics.csv",flat)
for row in flat:
    print(f'{row["partition"]}: {row["method"]}, {row["correct"]}/{row["n"]}, '
          f'macro-F1 {row["macro_f1"]:.3f}, false clearances {row["false_clearances"]}/{row["actual_present"]}')


Save confusion matrices and crops for every pair with a forced error. The selective SVM is shown separately. Neither the new scores nor these figures are used for model selection.

In [ ]:
def matrix_figure(group, entries):
    canvas = Image.new("RGB",(1030,275*len(entries)+58),"#f1f5fa")
    draw = ImageDraw.Draw(canvas)
    draw.text((20,12),group.capitalize()+" comparison | actual rows, predicted columns",
              font_size=25,fill="#172938")
    for i,(method,result) in enumerate(entries.items()):
        top=58+i*275
        draw.rounded_rectangle((12,top,1017,top+262),radius=13,fill="white")
        draw.text((30,top+12),method,font_size=25,fill="#172938")
        for j,label in enumerate(result["column_labels"]):
            draw.text((300+j*217,top+55),label.replace("_"," "),font_size=20,fill="#172938")
        for k,row in enumerate(result["matrix"]):
            ty=top+88+k*71
            draw.text((30,ty+16),LABELS[k].replace("_"," "),font_size=20,fill="#172938")
            for j,value in enumerate(row):
                px=286+j*217
                fill="#d8eee7" if j==k else "#f1dcd6" if value and j!=2 else "#edf1f7"
                draw.rounded_rectangle((px,ty,px+194,ty+54),radius=6,fill=fill)
                draw.text((px+87,ty+10),str(value),font_size=23,fill="#172938")
    canvas.save(RUN/(group+"_confusion.png"))

for group, entries in report.items():
    matrix_figure(group,entries)
    incorrect = {s["pair_id"] for s in scored[group] if s["prediction"] != s["truth"]}
    selected = [r for r in groups[group] if r["pair_id"] in incorrect]
    if not selected:
        continue
    sheet = Image.new("RGB",(950,157*len(selected)+35),"white")
    draw = ImageDraw.Draw(sheet)
    draw.text((8,5),group.capitalize()+" error crops | W left, inspection right",font_size=21,fill="black")
    with zipfile.ZipFile(ARCHIVE) as source:
        for i,row in enumerate(selected):
            top = 35+i*157
            mistakes = [s["method"] for s in scored[group]
                        if row["pair_id"]==s["pair_id"] and s["prediction"] != s["truth"]]
            draw.text((8,top+2),row["pair_id"],font_size=16,fill="black")
            for j,method in enumerate(mistakes):
                draw.text((8,top+25+19*j),method,font_size=15,fill="black")
            for j,key in enumerate(("W","inspection")):
                patch=Image.fromarray(np.round(255*crop(source,row,key)).astype(np.uint8))
                sheet.paste(ImageOps.contain(patch,(300,143)),(255+310*j,top))
    sheet.save(RUN/(group+"_error_crops.jpg"),quality=94)
print("Run artifacts:", RUN)
